# Tracking Evaluation

`DanceTrack val` で `YOLO11n + ByteTrack` と `YOLO11n + BoT-SORT` を比較する notebook です。

In [1]:
from __future__ import annotations

import base64
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    workspace_root = Path('/workspace')
    if (workspace_root / 'src').exists():
        PROJECT_ROOT = workspace_root
    else:
        for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if (candidate / 'src').exists() and (candidate / 'checkpoints').exists():
                PROJECT_ROOT = candidate
                break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.tracking.common import ensure_sequence_video, read_sequence_info

DANCETRACK_ROOT = PROJECT_ROOT / 'data' / 'dancetrack'
BYTETRACK_DIR = PROJECT_ROOT / 'checkpoints' / 'tracking' / 'bytetrack' / 'bytetrack_dancetrack_val'
BOTSORT_DIR = PROJECT_ROOT / 'checkpoints' / 'tracking' / 'bot_sort' / 'bot_sort_dancetrack_val'
GENERATED_VIDEO_ROOT = PROJECT_ROOT / 'notebooks' / 'tracking' / '.generated_videos'
VIDEO_FRAME_LIMIT = 600


In [2]:
def load_summary(experiment_dir: Path) -> dict:
    return json.loads((experiment_dir / 'best_test_results' / 'summary.json').read_text())


def load_sequence_metrics(experiment_dir: Path) -> pd.DataFrame:
    return pd.read_csv(experiment_dir / 'best_test_results' / 'sequence_metrics.csv')


def load_gt_stats(experiment_dir: Path) -> pd.DataFrame:
    return pd.read_csv(experiment_dir / 'best_test_results' / 'gt_sequence_stats.csv')


def select_joint_sequences(bytetrack_metrics: pd.DataFrame, botsort_metrics: pd.DataFrame, gt_stats: pd.DataFrame) -> dict[str, str]:
    merged = bytetrack_metrics.merge(botsort_metrics, on='sequence', suffixes=('_bytetrack', '_bot_sort'))
    merged = merged.merge(gt_stats, on='sequence', how='left')
    merged['avg_hota'] = merged[['HOTA_bytetrack', 'HOTA_bot_sort']].mean(axis=1)
    selected = {}
    used = set()

    def choose(df: pd.DataFrame):
        for seq in df['sequence'].tolist():
            if seq not in used:
                used.add(seq)
                return seq
        return None

    best = choose(merged.sort_values(['avg_hota', 'sequence'], ascending=[False, True]))
    if best:
        selected['best'] = best
    worst = choose(merged.sort_values(['avg_hota', 'sequence'], ascending=[True, True]))
    if worst:
        selected['worst'] = worst
    median_hota = merged['avg_hota'].median()
    middle = choose(merged.assign(median_distance=(merged['avg_hota'] - median_hota).abs()).sort_values(['median_distance', 'sequence']))
    if middle:
        selected['middle'] = middle
    hard = choose(merged.sort_values(['avg_objects_per_frame', 'max_objects_per_frame', 'sequence'], ascending=[False, False, True]))
    if hard:
        selected['hard'] = hard
    return selected


def ensure_comparison_videos(sequence_name: str, frame_limit: int = VIDEO_FRAME_LIMIT) -> dict[str, Path]:
    sequence = read_sequence_info(DANCETRACK_ROOT / 'val' / sequence_name)
    sequence_root = GENERATED_VIDEO_ROOT / sequence_name
    sequence_root.mkdir(parents=True, exist_ok=True)

    paths = {
        'gt': sequence_root / 'gt.mp4',
        'bytetrack': sequence_root / 'bytetrack.mp4',
        'bot_sort': sequence_root / 'bot_sort.mp4',
    }
    ensure_sequence_video(
        sequence=sequence,
        gt_output_path=paths['gt'],
        pred_output_path=paths['bytetrack'],
        prediction_txt_path=BYTETRACK_DIR / 'best_test_results' / 'mot_txt' / f'{sequence_name}.txt',
        frame_limit=frame_limit,
        tracker_video_name='bytetrack',
    )
    ensure_sequence_video(
        sequence=sequence,
        gt_output_path=paths['gt'],
        pred_output_path=paths['bot_sort'],
        prediction_txt_path=BOTSORT_DIR / 'best_test_results' / 'mot_txt' / f'{sequence_name}.txt',
        frame_limit=frame_limit,
        tracker_video_name='bot_sort',
    )
    return paths


def _video_data_uri(path: Path) -> str:
    payload = base64.b64encode(path.read_bytes()).decode('ascii')
    return f"data:video/mp4;base64,{payload}"


def render_three_videos(paths: dict[str, Path]) -> HTML:
    gt_uri = _video_data_uri(paths['gt'])
    bytetrack_uri = _video_data_uri(paths['bytetrack'])
    botsort_uri = _video_data_uri(paths['bot_sort'])
    html = f"""
    <div style='display:flex; gap:12px; align-items:flex-start;'>
      <div><div><b>GT</b></div><video width='420' controls preload='metadata'><source src='{gt_uri}' type='video/mp4'></video></div>
      <div><div><b>ByteTrack</b></div><video width='420' controls preload='metadata'><source src='{bytetrack_uri}' type='video/mp4'></video></div>
      <div><div><b>BoT-SORT</b></div><video width='420' controls preload='metadata'><source src='{botsort_uri}' type='video/mp4'></video></div>
    </div>
    """
    return HTML(html)


In [3]:
bytetrack_summary = load_summary(BYTETRACK_DIR)
botsort_summary = load_summary(BOTSORT_DIR)

summary_df = pd.DataFrame([
    {
        'experiment': 'bytetrack',
        'HOTA': bytetrack_summary['HOTA'],
        'IDF1': bytetrack_summary['IDF1'],
        'MOTA': bytetrack_summary['MOTA'],
        'IDs': bytetrack_summary['IDs'],
        'FP': bytetrack_summary['FP'],
        'FN': bytetrack_summary['FN'],
    },
    {
        'experiment': 'bot_sort',
        'HOTA': botsort_summary['HOTA'],
        'IDF1': botsort_summary['IDF1'],
        'MOTA': botsort_summary['MOTA'],
        'IDs': botsort_summary['IDs'],
        'FP': botsort_summary['FP'],
        'FN': botsort_summary['FN'],
    },
]).set_index('experiment')
summary_df


,HOTA,IDF1,MOTA,IDs,FP,FN
experiment,,,,,,
bytetrack,0.276108,0.279136,0.610283,4823,13310,69611
bot_sort,0.302460,0.289085,0.625056,4589,12549,67280


In [4]:
bytetrack_metrics = load_sequence_metrics(BYTETRACK_DIR)
botsort_metrics = load_sequence_metrics(BOTSORT_DIR)
gt_stats = load_gt_stats(BYTETRACK_DIR)

merged_metrics = bytetrack_metrics.merge(botsort_metrics, on='sequence', suffixes=('_bytetrack', '_bot_sort'))
merged_metrics = merged_metrics.merge(gt_stats, on='sequence', how='left')
merged_metrics['avg_hota'] = merged_metrics[['HOTA_bytetrack', 'HOTA_bot_sort']].mean(axis=1)
merged_metrics.sort_values('avg_hota', ascending=False).head(10)


,sequence,HOTA_bytetrack,DetA_bytetrack,AssA_bytetrack,IDF1_bytetrack,MOTA_bytetrack,FP_bytetrack,FN_bytetrack,IDs_bytetrack,HOTA_bot_sort,...,MOTA_bot_sort,FP_bot_sort,FN_bot_sort,IDs_bot_sort,seq_length,num_gt_boxes,num_gt_tracks,avg_objects_per_frame,max_objects_per_frame,avg_hota
5,dancetrack0018,0.542041,0.608401,0.486640,0.580633,0.698467,263,608,14,0.603983,...,0.693015,274,609,18,503,2935,8,5.834990,8,0.573012
24,dancetrack0097,0.363322,0.750648,0.176298,0.315353,0.896315,133,311,40,0.534979,...,0.909169,110,284,30,1203,4668,4,3.880299,4,0.449151
23,dancetrack0094,0.355730,0.483599,0.264762,0.406613,0.576134,345,4365,270,0.397693,...,0.587965,324,4280,237,603,11749,25,19.484245,23,0.376712
1,dancetrack0005,0.341536,0.689707,0.169615,0.319179,0.882043,132,355,72,0.379747,...,0.872758,162,372,69,1203,4739,4,3.939318,4,0.360642
18,dancetrack0073,0.346140,0.444291,0.275620,0.331746,0.395632,1174,3832,113,0.367122,...,0.398229,1241,3731,125,703,8470,14,12.048364,14,0.356631
13,dancetrack0043,0.371943,0.376344,0.371815,0.459305,0.447164,73,806,47,0.335593,...,0.465075,59,789,48,183,1675,14,9.153005,12,0.353768
15,dancetrack0058,0.341546,0.720582,0.162848,0.275862,0.852941,474,1027,124,0.364629,...,0.861448,449,970,112,1601,11050,7,6.901936,7,0.353087
3,dancetrack0010,0.326051,0.726459,0.146645,0.289947,0.892303,95,585,77,0.368962,...,0.892446,98,583,75,1203,7029,6,5.842893,6,0.347507
20,dancetrack0079,0.326114,0.503058,0.212900,0.383495,0.611995,626,4759,250,0.352347,...,0.618674,605,4679,254,1202,14523,18,12.082363,18,0.339230
2,dancetrack0007,0.289933,0.686862,0.123065,0.253365,0.836448,153,1263,133,0.317253,...,0.841939,134,1251,112,1203,9471,8,7.872818,8,0.303593


In [5]:
joint_selected_sequences = select_joint_sequences(bytetrack_metrics, botsort_metrics, gt_stats)
joint_selected_sequences


{'best': 'dancetrack0018',
 'worst': 'dancetrack0026',
 'middle': 'dancetrack0004',
 'hard': 'dancetrack0094'}